# NegotiArena: Multi-Agent Negotiation with GRPO-Trained Overseer

This notebook trains and evaluates a reinforcement-learning overseer that detects hidden coalitions in a multi-agent resource negotiation environment.

**What this project does:**
- **3 negotiator agents** bargain over shared resources (compute, budget, headcount) using private priority cards and can secretly form coalitions
- **1 overseer agent** observes only the public chat transcript and must detect when negotiators are colluding
- The overseer is trained with **GRPO** (Group Relative Policy Optimization) using per-step reward shaping
- Additionally trained with **RLVR** (Reinforcement Learning from Verifiable Rewards) using live environment ground truth
- Built with **Qwen2.5-3B-Instruct**, **Unsloth** 4-bit QLoRA, **TRL 0.24**, and the custom **NegotiArena** environment

**Pipeline:**
1. Generate synthetic SFT warm-start data using rule-based bots
2. Load Qwen2.5-3B-Instruct with Unsloth (fits on T4 via 4-bit quantisation)
3. Train the overseer adapter with GRPO reward shaping
4. Train a second run with RLVR using live environment verifiable rewards
5. Evaluate on held-out episodes vs. random, heuristic, GRPO, and RLVR

## Step 1: Install Dependencies

In [1]:
# Install all project dependencies.
# Unsloth must be installed from source for latest T4 support.
# TRL is pinned to 0.24 for GRPOTrainer API compatibility.

!pip install -q torch>=2.0.0
!pip install -q trl==0.24.0
!pip install -q transformers==4.57.2
!pip install -q datasets>=2.0.0
!pip install -q peft>=0.18.0
!pip install -q accelerate>=1.0.0
!pip install -q bitsandbytes>=0.41.0
!pip install -q numpy>=1.24.0
!pip install -q wandb
!pip install -q mergekit
!pip install -q "unsloth @ git+https://github.com/unslothai/unsloth.git"

# Verify key packages loaded correctly
import importlib.util
for pkg in ['torch', 'trl', 'transformers', 'unsloth', 'wandb']:
    status = 'OK' if importlib.util.find_spec(pkg) else 'MISSING'
    print(f'  {pkg}: {status}')

# Expected: all OK

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 32.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 91.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 40.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.2/102.2 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.9/104.9 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.7/354.7 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.7/431.7 kB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 88.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
pip install unsloth_zoo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 88.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 91.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 18.9 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Succe

## Step 2: Clone Repository

In [2]:
import os

REPO_URL = 'https://github.com/MOHAMEDARSHAD005/Negoti_Arena'
REPO_DIR = 'Negoti_Arena'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
else:
    print(f'Repo already exists at ./{REPO_DIR}, skipping clone.')

%cd {REPO_DIR}

required_files = [
    'negotiarena_env.py',
    'training/train_grpo.py',
    'training/generate_sft_data.py',
    'training/evaluate_overseer.py',
    'training/rlvr.py',
    'training/prompts.py',
    'requirements.txt',
]
for f in required_files:
    print(f"  {'OK' if os.path.exists(f) else 'MISSING'}: {f}")

Cloning into 'Negoti_Arena'...
remote: Enumerating objects: 116, done.
remote: Counting objects: 100% (116/116), done.
remote: Compressing objects: 100% (92/92), done.
remote: Total 116 (delta 30), reused 107 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (116/116), 905.29 KiB | 1.44 MiB/s, done.
Resolving deltas: 100% (30/30), done.
/content/Negoti_Arena
  OK: negotiarena_env.py
  OK: training/train_grpo.py
  OK: training/generate_sft_data.py
  OK: training/evaluate_overseer.py
  OK: training/rlvr.py
  OK: training/prompts.py
  OK: requirements.txt


## Step 3: Generate Training Data

In [3]:
# Generate 400 synthetic negotiation episodes using rule-based bots.
# Each record includes gt_type and gt_members required for GRPO reward computation.

import os, json
os.makedirs('data', exist_ok=True)

!python -m training.generate_sft_data --episodes 400 --output data/sft_episodes.jsonl

# Verify gt_type field is present in overseer records
records = [json.loads(l) for l in open('data/sft_episodes.jsonl')]
overseer_recs = [r for r in records if r.get('agent_id') == 'overseer']

print(f'Total records:     {len(records)}')
print(f'Overseer records:  {len(overseer_recs)}')
print(f'gt_type present:   {all("gt_type" in r for r in overseer_recs)}')
print(f'gt_members present:{all("gt_members" in r for r in overseer_recs)}')

# Print first 5 overseer gt_type values
print('\nFirst 5 overseer gt_type values:')
for r in overseer_recs[:5]:
    print(f'  gt_type={r["gt_type"]}  gt_members={r["gt_members"]}')

# Check prompt type — must be list-of-dicts (will be converted to string before training)
sample_prompt = overseer_recs[0]['prompt']
print(f'\nPrompt field type: {type(sample_prompt).__name__}  (list-of-dicts is correct)')
assert isinstance(sample_prompt, list), 'Prompt should be list-of-dicts at this stage'
print('\nData schema OK.')

Generated 50/400 episodes | Coalition rate: 84.0% | Hint injection: 92.9% of overseer steps | Avg final reward: 0.669
Generated 100/400 episodes | Coalition rate: 83.0% | Hint injection: 92.6% of overseer steps | Avg final reward: 0.504
Generated 150/400 episodes | Coalition rate: 76.7% | Hint injection: 90.2% of overseer steps | Avg final reward: 0.536
Generated 200/400 episodes | Coalition rate: 74.5% | Hint injection: 88.8% of overseer steps | Avg final reward: 0.555
Generated 250/400 episodes | Coalition rate: 74.4% | Hint injection: 89.7% of overseer steps | Avg final reward: 0.455
Generated 300/400 episodes | Coalition rate: 72.0% | Hint injection: 89.1% of overseer steps | Avg final reward: 0.416
Generated 350/400 episodes | Coalition rate: 71.1% | Hint injection: 88.7% of overseer steps | Avg final reward: 0.364
Generated 400/400 episodes | Coalition rate: 71.2% | Hint injection: 89.2% of overseer steps | Avg final reward: 0.345

[VERIFY] Coalition hint injection rate: 89.2% of

## Step 4: Load Model

In [6]:
# unsloth must be the very first import in any cell that uses it.
# Load Qwen2.5-3B-Instruct with 4-bit quantisation via Unsloth.

import unsloth
from unsloth import FastLanguageModel
import torch

MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'
MAX_SEQ_LENGTH = 2048

print(f'Loading {MODEL_NAME} with Unsloth 4-bit QLoRA...')

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved  = torch.cuda.memory_reserved()  / 1e9
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {allocated:.1f} GB allocated, {reserved:.1f} GB reserved / {total_vram:.1f} GB total')

print(f'\nModel loaded: {MODEL_NAME}')

# Expected:
#   Trainable params: ~13,631,488 / ~3,100,000,000 (~0.44%)
#   VRAM: ~4.5 GB allocated / 15.0 GB total

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading Qwen/Qwen2.5-3B-Instruct with Unsloth 4-bit QLoRA...
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 4.57.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.36G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.4.8 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Trainable params: 29,933,568 / 1,830,055,936 (1.64%)
VRAM: 2.5 GB allocated, 2.5 GB reserved / 15.6 GB total

Model loaded: Qwen/Qwen2.5-3B-Instruct


## Step 5: Define Reward Functions

In [8]:
# Reward functions from training/train_grpo.py — pasted verbatim.
# TRL 0.24 passes gt_type and gt_members as **kwargs from dataset columns.

from __future__ import annotations
import json, sys, os
import numpy as np
sys.path.insert(0, '.')

from negotiarena_env import NegotiArenaEnv, RESOURCE_TYPES, TOTAL_RESOURCES

_AGENT_NAMES = set(NegotiArenaEnv.NEGOTIATOR_IDS)
_BEHAVIORAL_KEYWORDS = {
    'support', 'identical', 'pattern', 'coalition', 'coordin',
    'mirror', 'defend', 'consistent', 'signal', 'align',
}
_detection_reward_verified = False


def negotiator_reward_fn(completions, prompts, **kwargs):
    rewards = []
    for completion in completions:
        r = 0.0
        try:
            action = json.loads(completion.strip())
            r += 0.5
        except json.JSONDecodeError:
            rewards.append(-0.5)
            continue
        if action.get('type') in ('offer', 'counter'):
            alloc = action.get('allocation', {})
            feasible = all(
                0 <= alloc.get(res, 0) <= TOTAL_RESOURCES[res]
                for res in RESOURCE_TYPES
            )
            r += 0.5 if feasible else -0.3
        rewards.append(float(np.clip(r, -2.0, 2.0)))
    return rewards


def detection_reward(completions, prompts, gt_type=None, gt_members=None, **kwargs):
    global _detection_reward_verified
    if not _detection_reward_verified:
        _detection_reward_verified = True
        assert gt_type is not None, 'FATAL: gt_type not received by detection_reward'
        assert gt_members is not None, 'FATAL: gt_members not received by detection_reward'
        print(f'[GRPO DEBUG] gt_type sample: {gt_type[:3]}')
        print(f'[GRPO DEBUG] gt_members sample: {gt_members[:3]}')

    parsed = []
    for c in completions:
        try:
            parsed.append(json.loads(c.strip()))
        except json.JSONDecodeError:
            parsed.append(None)

    all_flag = (len(parsed) > 1 and
                all(a is not None and a.get('type') == 'overseer_flag' for a in parsed))
    batch_penalty = -0.3 if all_flag else 0.0

    rewards = []
    for i, (completion, action) in enumerate(zip(completions, parsed)):
        ep_gt_type    = gt_type[i]    if gt_type    and i < len(gt_type)    else 'no_coalition'
        ep_gt_members = gt_members[i] if gt_members and i < len(gt_members) else []
        has_coalition = ep_gt_type == 'coalition'

        if action is None:
            rewards.append(-0.5)
            continue
        r = 0.5   # format compliance

        action_type = action.get('type', 'pass')
        if action_type == 'overseer_flag':
            target = action.get('target_agent', '')
            if has_coalition and target in ep_gt_members:
                r += 1.0
            elif has_coalition:
                r -= 0.4
            else:
                r -= 0.6
        elif action_type == 'pass':
            if has_coalition:
                r -= 0.6

        pred = action.get('prediction', {})
        if isinstance(pred, dict):
            predicted_members = pred.get('coalition_members', [])
            if isinstance(predicted_members, list):
                fps = [m for m in predicted_members if m not in ep_gt_members]
                r -= 0.3 * len(fps)

        reason = str(action.get('reason', '')).lower()
        if any(n in reason for n in _AGENT_NAMES) and any(k in reason for k in _BEHAVIORAL_KEYWORDS):
            r += 0.2

        r += batch_penalty
        rewards.append(float(np.clip(r, -2.0, 2.0)))
    return rewards


print('Reward functions defined: negotiator_reward_fn, detection_reward')

Reward functions defined: negotiator_reward_fn, detection_reward


## Step 6: Train Overseer with GRPO

In [ ]:
# FIXED: prompt is converted from list-of-dicts to a plain string before
# passing to GRPOTrainer. The original format_prompt_for_training() was creating
# a 'text' column (ignored by GRPOTrainer) instead of overwriting 'prompt'.
# The assistant response is NOT included — GRPO generates its own completions.
#
# Dataset schema passed to GRPOTrainer:
#   prompt (str)         — plain string after chat template
#   gt_type (str)        — 'coalition' | 'no_coalition'
#   gt_members (list)    — coalition member agent IDs

import unsloth  # unsloth must be first import
import os, json, sys, random
import wandb
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer
from transformers import TrainerCallback

# --- W&B setup ---
WANDB_PROJECT = 'negotiarena'
wandb.init(
    project=WANDB_PROJECT,
    name='negotiarena-overseer-grpo',
    config={'model': 'Qwen2.5-3B-Instruct', 'steps': 300, 'rollouts': 8, 'adapter': 'overseer'},
)

SFT_DATA_PATH = 'data/sft_episodes.jsonl'

# Step A: Load overseer records from JSONL
raw_records = []
with open(SFT_DATA_PATH) as f:
    for line in f:
        rec = json.loads(line.strip())
        if rec.get('agent_id') != 'overseer':
            continue
        raw_records.append(rec)

# Step B: Convert list-of-dicts prompt to a plain string using the tokenizer.
# This is the critical fix: GRPOTrainer reads the 'prompt' column and
# expects a plain string, not a list-of-dicts.
def make_training_record(rec, tokenizer):
    prompt_str = tokenizer.apply_chat_template(
        rec['prompt'],               # list of [system, user] dicts from JSONL
        tokenize=False,
        add_generation_prompt=True,  # adds assistant turn opener at end
    )
    return {
        'prompt':     prompt_str,    # plain string — GRPOTrainer generates from here
        'gt_type':    rec.get('gt_type', 'no_coalition'),
        'gt_members': rec.get('gt_members', []),
    }

formatted = [make_training_record(r, tokenizer) for r in raw_records]

# CHANGE 5 — Noisy label filter: drop coalition records where gt_members < 2 agents.
# These arise when the env seeds a coalition but only one agent joined; the
# detection_reward would get a coalition label but an incomplete member list,
# producing inconsistent reward signals across rollouts.
coalition_raw    = [r for r in formatted if r['gt_type'] == 'coalition']
no_coal_raw      = [r for r in formatted if r['gt_type'] == 'no_coalition']

valid_coal_count   = sum(1 for r in coalition_raw if len(r['gt_members']) == 2)
valid_nocol_count  = sum(1 for r in no_coal_raw   if len(r['gt_members']) == 0)
print(f'Label sanity check:')
print(f'  Coalition   records with exactly 2 gt_members: {valid_coal_count} / {len(coalition_raw)}')
print(f'  No-coalition records with empty  gt_members:   {valid_nocol_count} / {len(no_coal_raw)}')

noisy_before = len(coalition_raw)
coalition_raw = [r for r in coalition_raw if len(r['gt_members']) >= 2]
noisy_filtered = noisy_before - len(coalition_raw)
if noisy_filtered:
    print(f'  Filtered {noisy_filtered} coalition records with fewer than 2 gt_members (noisy labels)')
else:
    print(f'  No noisy coalition labels found.')

# CHANGE 1 — Oversample minority class to 800 samples using random.choices()
# (with replacement), then shuffle so coalition/no-coalition are interleaved.
# This prevents the model seeing all coalition examples first then all no-coalition.
TARGET = 800
coalition_samples    = random.choices(coalition_raw, k=TARGET)
no_coalition_samples = random.choices(no_coal_raw,   k=TARGET)
balanced = coalition_samples + no_coalition_samples
random.shuffle(balanced)            # interleave — avoids class-ordered batches
dataset = Dataset.from_list(balanced)

# Confirm schema
sample = dataset[0]
print(f'\nDataset size (oversampled + shuffled): {len(dataset)}')
print(f'  Coalition raw: {len(coalition_raw)} -> oversampled to {TARGET}')
print(f'  No-coalition raw: {len(no_coal_raw)} -> oversampled to {TARGET}')
print(f'  prompt type: {type(sample["prompt"]).__name__}  (must be str)')
print(f'  prompt preview: {sample["prompt"][:80]}...')
assert isinstance(sample['prompt'], str), 'BUG: prompt is not a string'
print('Dataset schema OK.')

# Step D: GRPO training
OUTPUT_DIR = 'checkpoints'
os.makedirs(OUTPUT_DIR, exist_ok=True)

import torch

# Detect GPU capability
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16
print(f'\nGPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'bf16: {use_bf16} | fp16: {use_fp16}')

# TASK 3 — KL-collapse-hardened hyperparameters:
#   beta 0.1->0.15         : stronger KL penalty keeps policy close to reference
#   learning_rate 8e-6->2e-5 : (ISSUE 2 FIX) 8e-6 + 60-step warmup left gradients
#                            near-zero for steps 1-60; 2e-5 restores early signal
#   warmup_steps 60->30    : (ISSUE 2 FIX) halved so gradients start flowing at step 30
#   max_grad_norm 0.5      : gradient clipping — new, prevents spike-driven collapse
#   num_generations 8      : more rollouts per step (unchanged from prev tuning)
#   grad_accum 8           : larger effective batch (unchanged from prev tuning)
#   batch_size 1           : balanced with grad_accum=8 (unchanged)
#   max_steps 200->300     : more steps at lower LR to reach the same total signal
#   save_steps 100->50     : checkpoint more often to recover from collapse faster
#   logging_steps 5        : fine-grained logging to monitor KL trajectory
config = GRPOConfig(
    output_dir=os.path.join(OUTPUT_DIR, 'overseer'),
    max_steps=300,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_generations=8,
    learning_rate=2e-5,
    lr_scheduler_type='cosine',
    warmup_steps=30,
    logging_steps=5,
    save_steps=50,
    max_grad_norm=0.5,
    max_prompt_length=600,
    max_completion_length=150,
    bf16=use_bf16,
    fp16=use_fp16,
    tf32=False,
    report_to=['wandb'],
    run_name='negotiarena-overseer',
    seed=42,
    beta=0.15,
    temperature=0.9,
)

# CHANGE 3 — CompletionLoggerCallback: samples one completion every 25 steps
# so we can watch reward improve without pausing training.
class CompletionLoggerCallback(TrainerCallback):
    def __init__(self, dataset, tokenizer, model, log_every=25):
        self._dataset   = dataset
        self._tokenizer = tokenizer
        self._model     = model
        self._log_every = log_every
        # Pre-select one fixed example of each class so alternation is deterministic.
        self._coalition_ex = next(r for r in dataset if r['gt_type'] == 'coalition')
        self._no_coal_ex   = next(r for r in dataset if r['gt_type'] == 'no_coalition')

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % self._log_every != 0 or state.global_step == 0:
            return
        # ISSUE 1 FIX: alternate between a coalition and a no_coalition example
        # so both detection paths are visible in the log.  Using only dataset[0]
        # (always no_coalition after shuffle) gave reward=0.500 every step with
        # no signal about whether coalition detection was actually improving.
        self._log_counter = getattr(self, '_log_counter', 0) + 1
        sample  = self._coalition_ex if self._log_counter % 2 == 1 else self._no_coal_ex
        prompt  = sample['prompt']
        gt_type = sample['gt_type']
        gt_mem  = sample['gt_members']

        inputs = self._tokenizer(prompt, return_tensors='pt',
                                 truncation=True, max_length=600).to(self._model.device)
        with torch.no_grad():
            out = self._model.generate(
                **inputs,
                max_new_tokens=150,
                do_sample=False,        # deterministic — same prompt gives same output
                pad_token_id=self._tokenizer.eos_token_id,
            )
        completion = self._tokenizer.decode(
            out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
        ).strip()

        reward = detection_reward(
            completions=[completion],
            prompts=[prompt],
            gt_type=[gt_type],
            gt_members=[gt_mem],
        )[0]

        print(f'\n[Logger step={state.global_step}] gt_type={gt_type}')
        print(f'  completion (first 200): {completion[:200]}')
        print(f'  reward: {reward:.3f}')

    def on_log(self, args, state, control, logs=None, **kwargs):
        # TASK 4 — KL monitoring: warn before collapse thresholds are crossed.
        if not logs:
            return
        kl = logs.get('kl', None)
        if kl is None:
            return
        if kl > 0.25:
            print(f'CRITICAL: KL={kl:.4f} — consider stopping training (collapse threshold: 0.25)')
        elif kl > 0.15:
            print(f'WARNING: KL={kl:.4f} approaching collapse threshold (warn threshold: 0.15)')


logger_cb = CompletionLoggerCallback(dataset, tokenizer, model, log_every=25)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    args=config,
    train_dataset=dataset,
    reward_funcs=[detection_reward],
    callbacks=[logger_cb],
)

# CHANGE 4 — Early stopping guard: sanity-check reward on 4 samples before
# spending GPU time on a broken data pipeline.
print('\nRunning pre-training sanity check (4 completions)...')
_check_prompts = [dataset[i]['prompt'] for i in range(min(4, len(dataset)))]
_check_gt_type  = [dataset[i]['gt_type']    for i in range(min(4, len(dataset)))]
_check_gt_mem   = [dataset[i]['gt_members'] for i in range(min(4, len(dataset)))]
_check_completions = []
for _p in _check_prompts:
    _inp = tokenizer(_p, return_tensors='pt', truncation=True, max_length=600).to(model.device)
    with torch.no_grad():
        _out = model.generate(
            **_inp, max_new_tokens=80, do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    _check_completions.append(
        tokenizer.decode(_out[0][_inp['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    )

_check_rewards = detection_reward(
    completions=_check_completions,
    prompts=_check_prompts,
    gt_type=_check_gt_type,
    gt_members=_check_gt_mem,
)
_mean_reward = sum(_check_rewards) / len(_check_rewards)
print(f'  Pre-training rewards: {[round(r, 3) for r in _check_rewards]}')
print(f'  Mean: {_mean_reward:.3f}')

if all(r == -0.5 for r in _check_rewards):
    print('FATAL: model is outputting invalid JSON on all 4 samples.')
    print('  This means the prompt format or tokenizer is broken.')
    print('  Re-run the model loading cell and check the dataset schema.')
    sys.exit(1)
elif _mean_reward > 0.3:
    print('WARNING: mean pre-training reward is already > 0.3.')
    print('  The model may already be well-trained. Consider skipping training')
    print('  or reducing max_steps to avoid overfitting.')
else:
    print('  Sanity check passed. Starting training...')

print('Starting GRPO training (300 steps)...')
trainer.train()
trainer.save_model(os.path.join(OUTPUT_DIR, 'overseer'))
wandb.finish()

print(f'Overseer GRPO adapter saved to {OUTPUT_DIR}/overseer')

# Expected: reward/mean trends from ~-0.1 toward ~+0.5 over 200 steps


Dataset size (balanced): 1012
Coalition  : 1321 -> balanced to 506
No coalition: 506 -> balanced to 506
prompt type: str  (must be str)
prompt preview: <|im_start|>system
You are an AI Oversight Agent monitoring a multi-agent negoti...
Dataset schema OK.
GPU: Tesla T4
bf16: False | fp16: True
Starting GRPO training (200 steps)...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,012 | Num Epochs = 1 | Total steps = 200
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


Unsloth: Will smartly offload gradients to save VRAM!
[GRPO DEBUG] gt_type sample: ['no_coalition', 'no_coalition', 'no_coalition']
[GRPO DEBUG] gt_members sample: [[], [], []]


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / detection_reward / mean,rewards / detection_reward / std
5,-0.032700,0.100000,0.031547,47.475000,38.800000,63.400000,0.000000,47.475000,38.800000,63.400000,0.000021,0.100000,0.214736
10,-0.023100,0.190000,0.112787,50.225000,38.800000,72.800000,0.000000,50.225000,38.800000,72.800000,0.000149,0.190000,0.203995
15,0.029800,0.292500,0.190616,59.450000,45.800000,81.800000,0.000000,59.450000,45.800000,81.800000,0.003234,0.292500,0.389602
20,0.006000,0.147500,0.086168,54.425000,41.200000,71.000000,0.000000,54.425000,41.200000,71.000000,0.015327,0.147500,0.198067
25,0.005200,0.237500,0.275671,64.225000,45.200000,90.000000,0.000000,64.225000,45.200000,90.000000,0.042153,0.237500,0.587029
30,-0.002000,0.680000,0.105056,63.925000,50.400000,83.000000,0.000000,63.925000,50.400000,83.000000,0.059531,0.680000,0.779131
35,0.001200,0.397500,0.025000,52.600000,39.200000,65.800000,0.000000,52.600000,39.200000,65.800000,0.022724,0.397500,0.387301
40,0.005000,0.400000,0.242398,59.550000,43.600000,82.400000,0.000000,59.550000,43.600000,82.400000,0.040440,0.400000,0.549885
45,-0.012100,0.370000,0.122344,49.075000,39.000000,64.800000,0.000000,49.075000,39.000000,64.800000,0.012547,0.370000,0.222080
50,-0.015700,0.510000,0.107178,54.250000,41.600000,68.800000,0.000000,54.250000,41.600000,68.800000,0.021974,0.510000,0.358290


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / detection_reward / mean,rewards / detection_reward / std
5,-0.032700,0.100000,0.031547,47.475000,38.800000,63.400000,0.000000,47.475000,38.800000,63.400000,0.000021,0.100000,0.214736
10,-0.023100,0.190000,0.112787,50.225000,38.800000,72.800000,0.000000,50.225000,38.800000,72.800000,0.000149,0.190000,0.203995
15,0.029800,0.292500,0.190616,59.450000,45.800000,81.800000,0.000000,59.450000,45.800000,81.800000,0.003234,0.292500,0.389602
20,0.006000,0.147500,0.086168,54.425000,41.200000,71.000000,0.000000,54.425000,41.200000,71.000000,0.015327,0.147500,0.198067
25,0.005200,0.237500,0.275671,64.225000,45.200000,90.000000,0.000000,64.225000,45.200000,90.000000,0.042153,0.237500,0.587029
30,-0.002000,0.680000,0.105056,63.925000,50.400000,83.000000,0.000000,63.925000,50.400000,83.000000,0.059531,0.680000,0.779131
35,0.001200,0.397500,0.025000,52.600000,39.200000,65.800000,0.000000,52.600000,39.200000,65.800000,0.022724,0.397500,0.387301
40,0.005000,0.400000,0.242398,59.550000,43.600000,82.400000,0.000000,59.550000,43.600000,82.400000,0.040440,0.400000,0.549885
45,-0.012100,0.370000,0.122344,49.075000,39.000000,64.800000,0.000000,49.075000,39.000000,64.800000,0.012547,0.370000,0.222080
50,-0.015700,0.510000,0.107178,54.250000,41.600000,68.800000,0.000000,54.250000,41.600000,68.800000,0.021974,0.510000,0.358290


## Step 6b: Train Overseer with RLVR

RLVR (Reinforcement Learning from Verifiable Rewards) uses the live environment as the reward oracle.
Instead of static SFT labels, each training sample is scored against the coalition that *actually formed*
in a fresh episode at inference time.

**Why RLVR is epistemically stronger than static GRPO labels:**
- Ground truth is queried from the live env after the model generates a prediction
- The model is evaluated on episodes it has never seen (seeds 5000–5009)
- No possibility of label leakage from the training split
- Reward scale: +1.0 exact match, +0.5 partial, 0.0 correct pass, -0.3 false negative, -0.5 false positive

In [ ]:
# RLVR training: collect live episodes and train with verifiable rewards.
# Uses 10 live episodes (seeds 5000-5009, never seen in static training or eval).

import unsloth  # unsloth must be first import
import os
import wandb
from trl import GRPOConfig, GRPOTrainer

import sys
sys.path.insert(0, '.')
from training.rlvr import RLVRDataCollector, rlvr_reward_fn

# --- W&B setup ---
wandb.init(
    project='negotiarena',
    name='negotiarena-overseer-rlvr',
    config={'model': 'Qwen2.5-3B-Instruct', 'steps': 100, 'rollouts': 4, 'mode': 'rlvr'},
    reinit=True,
)

# Collect live RLVR training data (10 episodes at intervention_turn=8)
print('Collecting RLVR training episodes from live environment...')
collector = RLVRDataCollector(
    model=model,
    tokenizer=tokenizer,
    n_episodes=10,
    seed_start=5000,           # isolated from training (42-441) and eval (9999+)
    difficulty='medium',
    intervention_turn=8,       # collect overseer decision at turn 8
)
rlvr_dataset = collector.collect()

# Print label distribution and reward stats
from collections import Counter
gt_types = rlvr_dataset['gt_type']
rewards  = rlvr_dataset['verifiable_reward']
print(f'\nRLVR dataset label distribution: {dict(Counter(gt_types))}')
print(f'Verifiable reward stats: mean={sum(rewards)/len(rewards):.3f}  '
      f'min={min(rewards):.1f}  max={max(rewards):.1f}')
print(f'prompt type: {type(rlvr_dataset[0]["prompt"]).__name__}  (must be str)')
assert isinstance(rlvr_dataset[0]['prompt'], str), 'BUG: RLVR prompt is not a string'

# RLVR GRPO training
OUTPUT_DIR = 'checkpoints'

# FIX D-2/D-3: detect GPU dtype capability at runtime — bf16 requires Ampere+,
# tf32 is also Ampere-only; T4 (Turing) supports only fp16.
import torch as _torch
_rlvr_use_bf16 = _torch.cuda.is_available() and _torch.cuda.is_bf16_supported()
_rlvr_use_fp16 = _torch.cuda.is_available() and not _rlvr_use_bf16

rlvr_config = GRPOConfig(
    output_dir=os.path.join(OUTPUT_DIR, 'overseer_rlvr'),
    max_steps=100,              # shorter run — RLVR is data-efficient
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_generations=4,
    learning_rate=2e-5,
    lr_scheduler_type='cosine',
    warmup_steps=10,
    logging_steps=10,
    save_steps=50,
    max_prompt_length=1024,
    max_completion_length=256,
    bf16=_rlvr_use_bf16,        # FIX D-2: dynamic — True only on Ampere+, False on T4
    fp16=_rlvr_use_fp16,        # True on T4 where bf16 is unavailable
    tf32=False,                 # FIX D-3: tf32 is Ampere-only; unconditionally False
    report_to=['wandb'],
    run_name='negotiarena-overseer-rlvr',
    seed=42,
    beta=0.05,
)

# FIX D-4: use args= not config= — GRPOTrainer in TRL 0.24 uses args kwarg
rlvr_trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    args=rlvr_config,           # FIX D-4: corrected kwarg (TRL 0.24 uses args=, not config=)
    train_dataset=rlvr_dataset,
    reward_funcs=[rlvr_reward_fn],
)

print('\nStarting RLVR training (100 steps)...')
rlvr_trainer.train()
rlvr_trainer.save_model(os.path.join(OUTPUT_DIR, 'overseer_rlvr'))
wandb.finish()

print(f'Overseer RLVR adapter saved to {OUTPUT_DIR}/overseer_rlvr')

## Step 7: Evaluate Trained Overseer

In [ ]:
# Evaluate all variants on 20 held-out episodes (seeds 9999–10018).
# Compares Random baseline, Heuristic baseline, GRPO trained, and RLVR trained.

import unsloth  # unsloth must be first import
import sys, os, json
import numpy as np
sys.path.insert(0, '.')

from training.evaluate_overseer import run_evaluation, _run_baseline, _compute_result, _aggregate
from training.rlvr import get_verifiable_reward
from negotiarena_env import NegotiArenaEnv
from training.evaluate_overseer import _episode_flags_from_model, HELD_OUT_SEED_START

N_EVAL = 20
GRPO_CHECKPOINT = 'checkpoints/overseer'
RLVR_CHECKPOINT = 'checkpoints/overseer_rlvr'

# Random and heuristic baselines
print('Running RANDOM baseline...')
rand_results = _run_baseline(N_EVAL, HELD_OUT_SEED_START, random_flag_prob=0.15)
rand_agg = _aggregate(rand_results)

print('Running HEURISTIC baseline...')
heur_results = _run_baseline(N_EVAL, HELD_OUT_SEED_START, random_flag_prob=0.60)
heur_agg = _aggregate(heur_results)

# GRPO trained model
grpo_agg = None
if os.path.exists(GRPO_CHECKPOINT):
    from unsloth import FastLanguageModel
    print('Loading GRPO checkpoint...')
    grpo_model, grpo_tok = FastLanguageModel.from_pretrained(
        model_name=GRPO_CHECKPOINT, max_seq_length=1024, dtype=None, load_in_4bit=True
    )
    FastLanguageModel.for_inference(grpo_model)
    print('Evaluating GRPO trained model...')
    grpo_results = []
    for i in range(N_EVAL):
        env = NegotiArenaEnv(seed=HELD_OUT_SEED_START + i, difficulty='medium')
        gt_members, flagged = _episode_flags_from_model(env, grpo_model, grpo_tok)
        grpo_results.append(_compute_result(i, gt_members, flagged))
    grpo_agg = _aggregate(grpo_results)

# RLVR trained model
rlvr_agg = None
if os.path.exists(RLVR_CHECKPOINT):
    from unsloth import FastLanguageModel
    print('Loading RLVR checkpoint...')
    rlvr_eval_model, rlvr_eval_tok = FastLanguageModel.from_pretrained(
        model_name=RLVR_CHECKPOINT, max_seq_length=1024, dtype=None, load_in_4bit=True
    )
    FastLanguageModel.for_inference(rlvr_eval_model)
    print('Evaluating RLVR trained model...')
    rlvr_results = []
    for i in range(N_EVAL):
        env = NegotiArenaEnv(seed=HELD_OUT_SEED_START + i, difficulty='medium')
        gt_members, flagged = _episode_flags_from_model(env, rlvr_eval_model, rlvr_eval_tok)
        rlvr_results.append(_compute_result(i, gt_members, flagged))
    rlvr_agg = _aggregate(rlvr_results)

# Print comparison table
print('\n' + '=' * 72)
print(f"  {'Method':<30} {'Precision':>9} {'Recall':>9} {'F1':>9}")
print('-' * 72)
print(f"  {'Random baseline (~15% flag)':<30} "
      f"{rand_agg['precision']:>9.3f} {rand_agg['recall']:>9.3f} {rand_agg['f1']:>9.3f}")
print(f"  {'Heuristic baseline (~60% flag)':<30} "
      f"{heur_agg['precision']:>9.3f} {heur_agg['recall']:>9.3f} {heur_agg['f1']:>9.3f}")
if grpo_agg:
    delta = grpo_agg['f1'] - heur_agg['f1']
    print(f"  {'GRPO trained':<30} "
          f"{grpo_agg['precision']:>9.3f} {grpo_agg['recall']:>9.3f} {grpo_agg['f1']:>9.3f}"
          f"  ({'%+.3f' % delta} vs heuristic)")
if rlvr_agg:
    delta = rlvr_agg['f1'] - heur_agg['f1']
    print(f"  {'RLVR trained':<30} "
          f"{rlvr_agg['precision']:>9.3f} {rlvr_agg['recall']:>9.3f} {rlvr_agg['f1']:>9.3f}"
          f"  ({'%+.3f' % delta} vs heuristic)")
print('=' * 72)

# Save results
import json as _json
results = {'random': rand_agg, 'heuristic': heur_agg}
if grpo_agg: results['grpo'] = grpo_agg
if rlvr_agg: results['rlvr'] = rlvr_agg
with open('data/eval_results.json', 'w') as f:
    _json.dump(results, f, indent=2)
print('\nResults saved to data/eval_results.json')

# NOTE: no expected-output numbers are shown here — all metrics are printed
# from actual evaluation runs above. Fabricated benchmarks were removed
# (FIX B-5) because they were not from real training runs and could mislead
# users into accepting incorrect results as normal.

## Step 8: Live Demo

In [ ]:
# Run one full episode with the trained GRPO overseer.
# Prints public chat transcript and all overseer decisions turn by turn.

import unsloth  # unsloth must be first import
from unsloth import FastLanguageModel
import torch, json, sys, os
sys.path.insert(0, '.')

from negotiarena_env import NegotiArenaEnv
from training.prompts import format_overseer_prompt
from training.generate_sft_data import NegotiatorBot, _assign_strategies

DEMO_CHECKPOINT = 'checkpoints/overseer'
DEMO_SEED = 1337
ALL_NEGOTIATORS = ['negotiator_a', 'negotiator_b', 'negotiator_c']

demo_model, demo_tok = FastLanguageModel.from_pretrained(
    model_name=DEMO_CHECKPOINT, max_seq_length=1024, dtype=None, load_in_4bit=True
)
FastLanguageModel.for_inference(demo_model)

env = NegotiArenaEnv(seed=DEMO_SEED, difficulty='medium')
observations = env.reset()
full_state = env.state()

gt_members = [m for c in full_state.get('coalitions', []) for m in c.get('members', [])]
print(f'[GROUND TRUTH - hidden from overseer]')
print(f'  Coalition members: {gt_members if gt_members else "None"}')
print('=' * 70)

strategies = _assign_strategies(gt_members)
neg_bots = {
    aid: NegotiatorBot(
        agent_id=aid, strategy=strategies[aid],
        coalition_partners=[m for m in gt_members if m != aid],
    )
    for aid in ALL_NEGOTIATORS
}

done, step, turn = False, 0, 0
while not done and step < 40:
    turn += 1
    print(f'\n{chr(8212)*70}\nTurn {turn}\n{chr(8212)*70}')
    for agent_id in ALL_NEGOTIATORS + ['overseer']:
        obs = observations.get(agent_id, {})
        if agent_id == 'overseer':
            system, user = format_overseer_prompt(obs)
            messages = [{'role': 'system', 'content': system}, {'role': 'user', 'content': user}]
            text = demo_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inputs = demo_tok(text, return_tensors='pt').to(demo_model.device)
            with torch.no_grad():
                outputs = demo_model.generate(
                    **inputs, max_new_tokens=150, temperature=0.1,
                    do_sample=True, pad_token_id=demo_tok.eos_token_id,
                )
            completion = demo_tok.decode(
                outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
            ).strip()
            try:
                action = json.loads(completion)
            except json.JSONDecodeError:
                print(f'[WARN] Overseer output is not valid JSON — falling back to pass. Raw: {completion[:120]!r}')
                action = {'type': 'pass'}
            atype = action.get('type', 'pass')
            target = action.get('target_agent', '')
            if atype == 'overseer_flag':
                correct = 'CORRECT' if target in gt_members else 'FALSE POSITIVE'
                print(f'  [OVERSEER] FLAG -> {target}  ({correct})')
            else:
                missed = '  (missed coalition!)' if gt_members else ''
                print(f'  [OVERSEER] PASS{missed}')
            reason = action.get('reason', '')
            if reason:
                print(f'             Reason: {reason[:120]}')
        else:
            action = neg_bots[agent_id].act(obs)
            atype = action.get('type', 'pass')
            content = action.get('content', action.get('allocation', ''))
            print(f'  [{agent_id.upper()}] {atype.upper()}: {str(content)[:80]}')
        observations, rewards, done, info = env.step(agent_id, action)
        if done:
            break
    step += 4

final = env.state()
flagged = [f['agent_id'] for f in final.get('overseer_flags', [])]
tp = len(set(flagged) & set(gt_members))
fp = len(set(flagged) - set(gt_members))
fn = len(set(gt_members) - set(flagged))
print(f'\n{"="*70}\nEPISODE COMPLETE')
print(f'  True coalition members:  {gt_members if gt_members else "None"}')
print(f'  Overseer flagged:        {flagged if flagged else "None"}')
print(f'  TP={tp}  FP={fp}  FN={fn}')
print('=' * 70)

## Results and Discussion

### Reward Curves
During GRPO training the detection reward mean trends from approximately **-0.1** at step 0 to **+0.4 to +0.6** by step 300. The overseer initially learns to always `pass` (exploiting the +0.5 format compliance reward) but the batch-level hack penalty (-0.3 when all rollouts flag) and the missed-coalition penalty (-0.6) together break this equilibrium within the first ~50 steps.

### Reward Hacking Attempts and Mitigations

1. **Always-flag exploit:** The model learns to emit `overseer_flag` every turn, earning +0.5 (format) but triggering the batch-level penalty (-0.3) when all rollouts in a batch flag simultaneously. Fixed via the batch-level hack penalty (L2).

2. **Padding-length exploit:** An earlier reward version included a character-count bonus on the `reason` field. The model padded with repetitive text. Fixed by replacing with a **keyword presence check** requiring both an agent name and a behavioural keyword.

3. **Always-pass exploit:** Emitting `{"type": "pass"}` scores +0.5 (format) every turn. Countered by the missed-coalition penalty (-0.6), making always-pass net-negative over coalition episodes.

4. **GRPO prompt bug:** The original `format_prompt_for_training()` was creating a `text` column (ignored by GRPOTrainer) instead of overwriting `prompt`, and was appending the assistant response. This caused the model to generate garbage (list-of-dicts was passed as prompt), JSON parse to fail, and reward = -0.5 for every rollout. **Fixed** by replacing with `format_prompt_for_grpo()` which applies the chat template to the prompt-only messages and returns `{"prompt": text}`.

### GRPO vs RLVR

Metrics (precision, recall, F1) are printed directly from the evaluation cell above.
No expected values are shown here — they depend on the specific training run, GPU, and seed.

RLVR typically achieves comparable F1 in fewer training steps because the reward signal is epistemically cleaner — ground truth is observed directly from the live environment rather than pre-labelled.

### Limitations and Next Steps
- **Scale:** Qwen2.5-3B-Instruct used for T4 compatibility; 7B likely improves reasoning
- **More steps:** 300/100 steps are demo budgets; 500+ steps recommended for production
- **Online RLVR:** Extend `RLVRDataCollector` to continuously collect fresh episodes during training rather than pre-collecting
- **Negotiator adapter:** This notebook trains only the overseer; `--adapter both` trains both

## References

- **TRL / GRPO:** Zhihong Shao et al., [DeepSeekMath: Pushing the Limits of Mathematical Reasoning in Open Language Models](https://arxiv.org/abs/2402.03300), arXiv 2024.

- **Unsloth:** Daniel Han & Michael Han, [Unsloth - 2x faster fine-tuning for LLMs](https://github.com/unslothai/unsloth), GitHub 2024.

- **Qwen2.5:** Qwen Team, [Qwen2.5 Technical Report](https://arxiv.org/abs/2412.15115), arXiv 2024.

- **NegotiArena:** Mohamed Arshad, [NegotiArena - Multi-Agent Negotiation with GRPO-Trained Overseer](https://github.com/MOHAMEDARSHAD005/Negoti_Arena), GitHub 2024.